# 智能体经典范式ReAct

## 什么是ReAct？
ReAct是一个结合了反思(Reflection)和行动(Action)的行为方式，它强调在行动过程中不断进行反思和调整，以实现更有效的学习和成长。ReAct的核心理念是通过不断地反思自己的行为和结果，来改进未来的行动，从而达到更好的效果从而去解决实际的问题。ReAct的流程通常包括以下几个步骤：
- Thoughts（思考）：在行动之前，先进行深入的思考，分析当前的情况、目标和可能的解决方案。
- Actions（行动）：根据思考的结果，采取具体的行动来实现目标。
- Observations（观察）：这是行动的返回结果，观察行动的效果和结果，收集相关的数据和反馈。

智能体会不断重复这个循环，将新的结果追加的历史记录中，形成一个不断迭代的上下文，直到Thoughts（思考）阶段的输出满足某个条件，或者达到预设的目标为止。这个过程形成了一个强大的协同效应，使得智能体能够在不断的反思和行动中不断优化自己的行为，从而更有效地解决问题和实现目标。

## 适合那些场景？
ReAct适用于需要持续学习和适应的场景，特别是在复杂和动态的环境中。以下是一些适合ReAct的场景：
- 需要外部知识的场景：如查询实时信息、搜索专业领域的只是。
- 需要精确计算：如数学问题交给计算器工具避免大模型计算错误。
- 需要与API交互：如调用外部API获取数据或执行特定任务。

## 工具的定义与实现

如果说大模型是智能体的大脑，那么工具(Tools)就是智能体的手脚。工具是智能体用来与外界交互、获取信息或执行任务的手段。工具可以是各种形式的接口、API、数据库查询、计算器等，帮助智能体在思考和行动过程中更有效地获取信息和执行任务。

我们来实现一个外部搜索工具，它将为智能体提供一个网页搜索工具，允许智能体在需要时查询实时信息。

In [1]:
# 安装谷歌搜索工具包
!pip install google-search-results

同时，你需要前往 SerpApi官网 注册一个免费账户，获取你的API密钥，并将其添加到我们项目根目录下的 .env 文件中：

In [2]:
# .env file
# ... (保留之前的LLM配置)
SERPAPI_API_KEY="YOUR_SERPAPI_API_KEY"

### 定义一个良好的工具接口三要素
1. **工具名称**：每个工具都应该有一个独特的名称，以便智能体能够识别和调用它。
2. **工具描述**：每个工具都应该有一个清晰的描述，说明它的功能和用途，以帮助智能体理解何时使用它，`这是整个机制中最重要的一部分`。
3. **工具函数**：工具的具体实现函数，定义了工具的输入参数和输出结果，智能体通过调用这个函数来执行工具的功能。

In [3]:
from dotenv import load_dotenv

load_dotenv()
import os
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f"🔍 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }

        client = SerpApiClient(params)
        results = client.get_dict()

        # 智能解析:优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)

        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"


### 定义通用的工具执行器

当智能体需要调用工具时，我们需要一个通用的工具执行器来处理工具的调用和结果返回。这个执行器将负责接收智能体的工具调用请求，执行相应的工具函数，并将结果返回给智能体。

In [4]:
from typing import Dict, Any, Callable, Optional

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: Callable) -> None:
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> Optional[Callable]:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])

# 避免静态分析器提示变量未定义（示例中后面会初始化或填充）
toolExecutor = None


### 测试工具执行器
我们来测试一下工具执行器，看看它是否能够正确地注册和调用工具。

In [5]:
# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的实战搜索工具
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)

    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: Search['英伟达2026最新的GPU型号是什么'] ---")
    tool_name = "Search"
    tool_input = "英伟达2026最新的GPU型号是什么"

    tool_function = toolExecutor.getTool(tool_name)
    if tool_function:
        observation = tool_function(tool_input)
        print("--- 观察 (Observation) ---")
        print(observation)
    else:
        print(f"错误:未找到名为 '{tool_name}' 的工具。")


工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 执行 Action: Search['英伟达2026最新的GPU型号是什么'] ---
🔍 正在执行 [SerpApi] 网页搜索: 英伟达2026最新的GPU型号是什么
--- 观察 (Observation) ---
[1] 消息称英伟达调整2026年RTX 50系列显卡：力推8GB版
消息源指出英伟达已减少RTX 5060 Ti 16GB 和RTX 5070 Ti 16GB 两款高显存型号的出货量，未来将把核心资源集中在RTX 5060 和RTX 5060 Ti 8GB 这两个型号上。

[2] 独显已死？英伟达PC芯片2026年登场，GPU性能对标RTX ...
过去一年多，很多信息其实都得到了披露。代号N1/N1X，基于Arm 架构，集成Blackwell 架构GPU，为Windows on Arm 定制，采用统一内存设计，定位是强调AI 与 ...

[3] 消息称英伟达调整2026 年RTX 50 系列显卡：力推8GB 显存版
NVIDIA 以及AIC 品牌厂商将重新调整为：RTX 5060、RTX5060TI 8G 系列两个系列，将成为主力+ 物流明显产品，也确定是NV RTX 50 系列中国市场重点销售的两个 ...


## ReAct智能体的定义与实现
接下来，我们将定义一个ReAct智能体类，这个类将结合我们之前定义的工具执行器，来实现ReAct智能体的核心功能。这个智能体将能够在思考(Thoughts)阶段分析问题，在行动(Actions)阶段调用工具，并在观察(Observations)阶段处理工具的返回结果。

### (1).提示词设计
在ReAct智能体中，提示词设计是非常重要的一环。我们需要设计一个清晰的提示词模板，来指导智能体在思考、行动和观察阶段的行为。

In [6]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

这个模板定义了智能体和LLM交互的规范：
- 角色定义：明确智能体是一个有能力调用外部工具的智能助手。
- 工具列表：提供一个动态生成的工具列表，帮助智能体了解可用的工具和它们的功能。
- 格式规范：规定智能体必须按照特定的格式输出Thought和Action，确保LLM能够正确解析智能体的思考和行动
- 动态上下文：通过Question和History字段，智能体可以获取当前的问题和之前的交互历史，帮助它更好地进行思考和决策。

### 循环核心的实现

ReActAgent的核心是一个循环，在这个循环中，智能体会不断地进行思考、行动和观察，直到满足某个条件（如达到预设的目标或输出最终答案）为止。这个循环的实现需要处理智能体与LLM之间的交互，以及工具调用和结果处理的逻辑。


In [7]:
import re
from utils.llm import HelloAgentsLLM


class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        运行ReAct智能体来回答一个问题。
        """
        self.history = [] # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM进行思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)

            if not response_text:
                print("错误:LLM未能返回有效响应。")
                break

            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)

            if thought:
                print(f"思考: {thought}")

            if not action:
                print("警告:未能解析出有效的Action，流程终止。")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer

            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... 处理无效Action格式 ...
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")

            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input) # 调用真实工具
                print(f"👀 观察: {observation}")

            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        print("已达到最大步数，流程终止。")
        return None

    def _parse_output(self, text: str):
        """解析LLM的输出，提取Thought和Action。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """解析Action字符串，提取工具名称和输入。
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None



### ReActAgent 执行逻辑详解（逐步解析）

下面通过“契约 / 输入 / 输出 / 关键步骤 / 错误与边界情况”的结构，详细说明 `ReActAgent` 的运行机制，便于阅读与二次开发：

1) 契约（Contract）
- 输入：
  - question: str — 用户的问题或任务描述。
  - llm_client: HelloAgentsLLM — 提供 think(messages) 方法的 LLM 客户端。
  - tool_executor: ToolExecutor — 已注册工具的执行器（通过 name->func 映射获取工具）。
  - max_steps: int — 最大循环步数，用于防止无限循环。
- 输出：
  - 返回最终答案字符串（当遇到 Finish[...] 行为时）。
  - 若达到最大步数或发生错误，返回 None。

2) 数据形态与历史（History）
- self.history: List[str] — 以字符串形式追加每一轮的 Action 和 Observation（例如："Action: Search[query]" / "Observation: ..."）。
- prompt 由 REACT_PROMPT_TEMPLATE、可用工具描述、问题和 history 拼接生成，传给 LLM 进行思考。

3) 逐步执行流程（Step-by-step）
- 进入循环（最多 max_steps 次）
  1. 增量步数计数 current_step += 1，并打印当前步数。
  2. 通过 tool_executor.getAvailableTools() 获取工具列表描述，会注入到提示词（prompt）中。
  3. 将历史（history）拼接成 history_str，一并传入提示词，形成上下文。
  4. 调用 llm_client.think(messages=messages) 获取 LLM 的响应（response_text）。
  5. 使用 _parse_output(response_text) 从 LLM 响应中提取 Thought 和 Action 两个部分：
     - Thought 用于调试/打印，不用于驱动执行逻辑（但在后续版本可用于内部决策）。
     - Action 必须符合约定：要么是工具调用格式 name[input]，要么是 Finish[最终答案]。
  6. 如果 Action 以 "Finish" 开头：使用正则提取最终答案并返回，结束循环。
  7. 否则，解析工具名称和输入（_parse_action），若解析失败则跳过本轮或记录警告。
  8. 通过 tool_executor.getTool(tool_name) 获取工具函数并调用，获得 observation（观察）。
  9. 将本轮的 Action 与 Observation 追加到 self.history 中，用于下次提示增强上下文。
  10. 循环继续直到返回答案或达到 max_steps。

4) 关键实现细节
- 提示词模板：REACT_PROMPT_TEMPLATE 强制 LLM 输出包含 Thought 和 Action 两部分，便于解析。
- 输出解析：_parse_output 使用正则（支持 DOTALL）来抓取多行的 Thought 与 Action，注意要兼容 LLM 可能包含换行的输出。
- Action 解析：_parse_action 支持匹配形如 ToolName[...任意文本...] 的格式，使用 (\w+) 捕获工具名（字母/数字/下划线），和 (.*) 捕获内部内容（允许多行）。
- 工具调用：工具执行器只保存 name->func 映射；当找不到工具时，会把错误信息放入 observation 并记录到历史，避免崩溃。

5) 错误处理与边界情况
- LLM 无响应：若 response_text 为空，打印错误并终止流程（返回 None）。
- 无法解析 Action：若从 LLM 输出中无法提取 Action，会打印警告并终止以避免不可控行为。
- 工具不存在：当 getTool 返回 None，会生成错误观察并继续（可根据需求改为抛错或终止）。
- 非法 Action 格式：若 _parse_action 无法解析 name 或 input，会跳过该轮（当前实现是 continue）；可改为记录并终止。
- 最大步数保护：max_steps 用于防止无限循环；开发者可根据任务复杂度调整该值或改为更复杂的停机条件。



# 确保在示例运行时有一个可用的 toolExecutor（防止 NameError）
try:
    toolExecutor
except NameError:
    toolExecutor = ToolExecutor()
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)


In [11]:
react = ReActAgent(llm_client=HelloAgentsLLM(), tool_executor=toolExecutor)
question = "成龙的真实姓名是什么？"
react.run(question)

--- 第 1 步 ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
Thought: 用户询问成龙的真实姓名。我知道成龙是一位著名的香港演员，他的真实姓名是房仕龙，艺名是成龙。但为了确保信息的准确性和时效性，我应该使用搜索工具进行核实。

Action: Search[成龙真实姓名]
思考: 用户询问成龙的真实姓名。我知道成龙是一位著名的香港演员，他的真实姓名是房仕龙，艺名是成龙。但为了确保信息的准确性和时效性，我应该使用搜索工具进行核实。
🎬 行动: Search[成龙真实姓名]
🔍 正在执行 [SerpApi] 网页搜索: 成龙真实姓名
👀 观察: [1] 成龙- 维基百科，自由的百科全书
陈港生，SBS，MBE，PMW（英语：Jackie Chan，1954年4月7日—），谱名房仕龙，艺名成龙、元楼，祖籍今安徽芜湖，香港男演员、歌手、导演、编剧和电影监制。1979年加盟嘉禾公司，填补 ...

[2] 成龍- 維基百科，自由的百科全書
陳港生，SBS，MBE，PMW（英語：Jackie Chan，1954年4月7日—），譜名房仕龍，藝名成龍、元樓，祖籍今安徽蕪湖，香港男演員、歌手、導演、編劇和電影監製。1979年加盟嘉禾公司，填補 ...

[3] 为啥成龙姓陈，他儿子姓房？
这个问题说来简单，因为成龙自己其实也不姓龙，而是姓陈，本名叫陈港生。 但陈也不是他的本姓，祖上其实姓房，之所以姓陈，跟他的父亲有关。 成龙的父亲名 ...
--- 第 2 步 ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
Thought: 用户的问题是“成龙的真实姓名是什么？”。根据提供的搜索历史，我已经调用了搜索引擎，并且返回了结果。观察结果显示，成龙的谱名（家族名）是房仕龙，他的本名是陈港生，艺名是成龙。问题问的是“真实姓名”，这通常指的是本名或出生时登记的名字。根据搜索结果，他的本名是陈港生。同时，信息也提到了他的谱名房仕龙。为了确保准确性，我需要确认哪个是他的法定真实姓名。搜索结果[1]和[2]明确指出“陈港生”是他的名字，而“房仕龙”是谱名。在中文语境下，“真实姓名”通常指身份证或官方文件上的名字，即陈港生。我已经有了足够的信息来回答这个问题。

Action: Finish[成龙的真实姓名是陈港

'成龙的真实姓名是陈港生。他的谱名（家族名）是房仕龙，艺名是成龙。'

## ReActAgent的特点局限性和调试技巧
### 特点
1. **高可解释性**：ReActAgent通过Thought和Action的明确分离，使得智能体的思考过程和行动决策更加透明，便于开发者理解和调试。
2. **动态规划能力**：智能体可以根据观察结果不断调整自己的行动策略，实现动态规划和迭代优化。
3. **工具调用能力**：通过工具执行器，ReActAgent能够灵活地调用外部工具，扩展了智能体的能力范围，适用于需要实时信息查询或复杂计算的场景。

### 局限性
1. **依赖LLM的输出质量**：ReActAgent的性能高度依赖于LLM的输出质量，如果LLM生成的Thought或Action不符合预期，可能会导致智能体无法正确执行任务。
2. **执行效率问题**：在复杂任务中，智能体可能需要多次循环才能达到目标，这可能会导致执行效率较低，尤其是在每次调用LLM时的延迟，以及成本。
3. **提示词设计敏感性**：ReActAgent的表现对提示词设计非常敏感，不同的提示词可能会导致智能体产生不同的行为，设计不当可能会导致智能体无法正确理解任务或工具的使用。
4. **可能陷入局部最优**：智能体在迭代过程中可能会陷入局部最优解，无法找到全局最优的解决方案，尤其是在复杂问题空间中。

### 调试技巧
- **逐步调试**：在每个循环步骤中打印Thought、Action和Observation，帮助开发者理解智能体的思考过程和决策逻辑。
- **分析LLM输出**：如果智能体的行为不符合预期，检查LLM的输出是否正确解析了Thought和Action，确保LLM生成的内容符合提示词的要求。
- **验证工具调用**：确保工具执行器正确注册了工具，并且工具函数能够正确执行，特别是在处理输入和输出时。
- **调整提示词**：如果智能体的行为不理想，尝试调整提示词的设计，增加更多的指导信息或示例，帮助LLM更好地理解任务和工具的使用。
- **限制循环次数**：在调试过程中，可以适当减少max_steps的值，以便更快地观察智能体的行为，避免长时间的循环等待。
- **尝试不同的模型和参数**：如果LLM的输出质量不佳，可以尝试使用不同的模型或调整生成参数（如温度、最大长度等）来改善输出质量。